### ***Interactive Dashboard***

In [35]:
import pandas as pd
import numpy as np
from dash import Dash,html,dcc,callback,Output,Input
import plotly.express as px

In [36]:
# read data
dataframe=pd.read_csv(r"https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv")

In [37]:
launch_sites=dataframe['Launch Site'].unique()
result=[{'label': 'All Sites', 'value': 'ALL'}]
for site in launch_sites:
    result.append({'label': site, 'value': site})
    


In [49]:
filtered_df=dataframe[dataframe['Launch Site']=='CCAFS LC-40']
filtered_df=filtered_df.groupby('class').count()
filtered_df
fig=px.pie(filtered_df,names=filtered_df.index,values='Launch Site',title=f'Success rate')
fig

In [59]:
min_value=min(dataframe['Payload Mass (kg)'])
max_value=max(dataframe['Payload Mass (kg)'])

In [60]:
max_value

9600.0

In [89]:
app=Dash()
app.layout=[
   html.H1(children='SpaceX Launch Records Dashboard',style={'textAlign':'center'}),
   dcc.Dropdown(id='site-dropdown',
                  options=result,
                  value='All',
                  placeholder='Select Launch Site',
                  searchable=True
       

   ),
   dcc.RangeSlider(id='payload-slider',min=0,max=10000,step=1000,marks={0:'0',100:'100'},value=[min_value,max_value]),
   dcc.Graph(id='success-pie-chart'),
   dcc.Graph(id='success-payload-scatter-chart')

   


]

@app.callback(Output(component_id='success-pie-chart',component_property='figure'),Input(component_id='site-dropdown',component_property='value'))
def get_pie_chart(entered_site):
    filtered_df=dataframe[dataframe['Launch Site']==entered_site]
    filtered_df=filtered_df.groupby('class').count()
    if entered_site=='ALL':
        fig=px.pie(dataframe,values='class',names='Launch Site',title='All sites output')
        return fig
    else :
        fig=px.pie(filtered_df,values='Launch Site',names=filtered_df.index,title=f'Success rate of {entered_site}')
        return fig

@app.callback(Output(component_id='success-payload-scatter-chart',component_property='figure'),[Input(component_id='site-dropdown',component_property='value'),Input(component_id='payload-slider',component_property='value')])
def success_payload_scatter(entered_site,payload):
    data=dataframe[(dataframe['Launch Site']==entered_site)]
    # data=dataframe[dataframe['Payload Mass (kg)']>payload]
    if entered_site=='All':
        fig=px.scatter(dataframe,x='Payload Mass (kg)',y='class',color='Booster Version Category',title='Relation between payload mass and class')
        return fig
    else:
        fig=px.scatter(data,x='Payload Mass (kg)',y='class',color='Booster Version Category',title=f'Relation between {entered_site} having payload pass {payload[1]} kg')
        return fig

In [90]:
dataframe.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [91]:
if __name__=='__main__':
    app.run(debug=True)